In [3]:
# --- TASK 1: REINFORCEMENT LEARNING ---
import numpy as np
import random
#define maze environment

class MazeEnvironment:
    def __init__(self):
        # grid dimensions and Start/Goal positions defined according to 
        #cw spec, Dracopoulos, D. (2025) 6ELEN018W Applied Robotics Coursework Specification 
        # 0 is empty, 1 is obstacle
        self.rows = 6
        self.cols = 6
        self.grid = np.array([
            [0, 0, 1, 0, 1, 0],
            [0, 0, 0, 0, 1, 0],
            [0, 1, 1, 0, 0, 0],
            [0, 0, 0, 1, 0, 0],
            [0, 0, 0, 0, 1, 0],
            [0, 0, 1, 0, 0, 0]
        ])
        # key of actions: 0 is north, 1 is east, 2 is south, 3 is west
        self.actions =  [0,1,2,3]
        self.action_names = {0: 'N', 1: 'E', 2: 'S', 3: 'W'}
    
    def is_valid(self, r, c):
    # to check if a coordinate is within bounds and NOT an obstacle
        if 0 <= r < self.rows and 0 <= c < self.cols:
            if self.grid[r, c] == 0:
                return True
            return False
    def get_valid_cells(self):
        # to return a list of all coordinates (r,c) that are NOT obstacles
        cells = []
        for r in range(self.rows):
            for c in range(self.cols):
                if self.grid[r, c] == 0:
                    cells.append((r, c))
        return cells
    
    def step(self,state,action):
    #apply action to state (r,c), returns next_state.
    #if hit wall or obstacle, it stays in the same cell!
        r,c = state
        #calculate possible new coordinates
        if action == 0: # north
            nr, nc = r - 1, c
        elif action == 1: #east
            nr, nc = r, c + 1
        elif action == 2: #south
            nr, nc = r + 1, c
        elif action == 3: #west
            nr, nc = r, c - 1
        else:
            return state
            
        #checking validity
        if self.is_valid(nr, nc):
            return (nr, nc)
        else:
            return state #stay same
# implementing q-learning

def run_q_learning(env, goal_pos, gamma=0.9, episodes=1000):
    # running q-learning for a speciic goal position
    #based on the update rule: Q(s,a)=r+gamma*max(Q(s',a'))
    # logic adapted from Reinforcement Learning Exercises, Dracopoulos, D. (2025) Tutorial 9
    
    #intialise q-table
    # numpy array Q[row, col, action]
    Q = np.zeros((env.rows, env.cols, 4))
    
    valid_cells = env.get_valid_cells()
    
    for e in range(episodes):
        #starting from random valid position
        start_idx = random.randint(0, len(valid_cells) - 1)
        current_state = valid_cells[start_idx]
        
        #loop until goal reached (limit has been set to 1000 to avoid infinite loops!)
        steps = 0
        while current_state != goal_pos and steps < 1000:
            r,c = current_state
            action = random.choice(env.actions)
            next_state = env.step(current_state, action)
            nr, nc = next_state
            
            #give reward
            
            if next_state == goal_pos:
                reward = 100
            
            else:
                reward = 0
                
            #updating q-value
            max_q_next = np.max(Q[nr, nc, :])
            Q[r, c, action] = reward + gamma * max_q_next
            #move to next
            current_state = next_state
            steps += 1
        return Q